In [18]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [ ]:
pip install dagshub mlflow --quiet

In [4]:
import dagshub
import mlflow
import mlflow.sklearn

dagshub.init(repo_owner='tvani2', repo_name='titanic_tutoring', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=0331aefd-5493-44d9-b716-744bf85bbfd2&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e868cff1134b5660af7800f79b7b8167f58015c607b6144e0e5a9acc7ace0b96




Accessing as tvani2

Initialized MLflow to track repo "tvani2/titanic_tutoring"

Repository tvani2/titanic_tutoring initialized!

In [13]:
preprocessing_run_id = "48928553183941d8975d256fbbf20d86" 
best_run_id          = "236d2446c96941e2954186ef9c719c55"

# load preprocessing pipeline
trusted_types = [
    "__main__.FamilySizeAdder",
    "__main__.WOEEncoder",
    "__main__.ColumnNameRestorer",
    "__main__.RFESelector",
    "__main__.CorrelationFilter",
    "__main__.IVValidator",
    "sklearn.pipeline.Pipeline",
    "sklearn.compose._column_transformer.ColumnTransformer",
    "sklearn.compose._column_transformer._RemainderColsList",
    "sklearn.impute._base.SimpleImputer",
    "sklearn.preprocessing._encoders.OneHotEncoder",
    "sklearn.linear_model._logistic.LogisticRegression",
    "sklearn.feature_selection._rfe.RFE",
    "numpy.dtype",
]

pipeline_path = mlflow.artifacts.download_artifacts(
    run_id=preprocessing_run_id,
    artifact_path="preprocessing_pipeline.skops"
)
preprocessing_pipeline = sio.load(pipeline_path, trusted=trusted_types)
print("Pipeline loaded!")

# load best model
model_path = mlflow.artifacts.download_artifacts(
    run_id=best_run_id,
    artifact_path="model.skops"
)
best_model = sio.load(model_path, trusted=trusted_types)
print("Model loaded!")
print(type(best_model))

Pipeline loaded!


Model loaded!
<class 'sklearn.linear_model._logistic.LogisticRegression'>


In [20]:
test_df = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
passenger_ids = test_df["PassengerId"]

X_kaggle = preprocessing_pipeline.transform(test_df)
kaggle_predictions = best_model.predict(X_kaggle)

submission = pd.DataFrame({
    "PassengerId": passenger_ids,
    "Survived":    kaggle_predictions
})
submission.to_csv("submission.csv", index=False)

print(submission.shape)                # (418, 2)
print(submission["Survived"].unique()) # [0 1]
print(submission.head())

(418, 2)
[0 1]
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1
